# Lección 5.4 — Árboles de decisión: el juego de las 20 preguntas

> **Módulo 5 · Modelos clásicos supervisados** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.2 (función de pérdida), Lección 4.4 (sobreajuste vs. subajuste)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía cómo un árbol de decisión toma una decisión.
- Describir **cómo aprende** un árbol: cómo elige qué pregunta hacer y en qué orden.
- Predecir el efecto de los hiperparámetros principales (`max_depth`, `min_samples_leaf`…) sobre el sobreajuste.
- Reconocer cuándo un árbol es una buena elección y cuándo no.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import make_moons, load_breast_cancer

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el juego de las 20 preguntas

¿Has jugado a adivinar un animal haciendo solo preguntas de sí o no?

```
¿Vive en el agua?
 ├── Sí → ¿Es un mamífero?
 │         ├── Sí → 🐬 Delfín
 │         └── No → 🐟 Pez
 └── No → ¿Puede volar?
           ├── Sí → 🦅 Águila
           └── No → 🐘 Elefante
```

Un buen jugador no empieza preguntando *"¿Es un ornitorrinco?"*. Empieza con preguntas que **dividen las posibilidades lo más posible**, porque cada respuesta descarta muchas opciones de golpe.

**Un árbol de decisión hace exactamente eso con los datos:** aprende *qué* preguntas hacer, y *en qué orden*, para llegar a la respuesta correcta con el menor número de preguntas posible.

### ✋ Ejercicio 1 — Calentamiento
Quieres adivinar un número secreto entre 1 y 100. ¿Cuál es mejor primera pregunta?
- a) "¿Es el 37?"
- b) "¿Es mayor que 50?"
- c) "¿Es múltiplo de 7?"

¿Por qué? Escribe tu respuesta en la celda de abajo antes de mirar la solución.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**b)** Pase lo que pase, descarta la **mitad** de las posibilidades. La a) casi siempre descarta solo una opción; la c) parte el grupo de forma muy desigual (14 sí / 86 no).

Esta es exactamente la idea que guía a un árbol: **elegir la pregunta que mejor divide**.
</details>

---
## 2. 📖 El concepto: un árbol es una secuencia de preguntas

Un árbol de decisión predice algo haciendo una **secuencia de preguntas sobre las características** (*features*) de un ejemplo.

| Término | Qué es | En la analogía |
|---|---|---|
| **Nodo raíz** | La primera pregunta | "¿Vive en el agua?" |
| **Nodo interno** | Cualquier pregunta intermedia | "¿Puede volar?" |
| **Rama** | Una posible respuesta | "Sí" / "No" |
| **Hoja** | El final del camino: la predicción | 🐬 Delfín |
| **Profundidad** | Número de preguntas en el camino más largo | 2 |

Vamos a verlo con datos. Un banco tiene el historial de 300 clientes y quiere decidir si **aprueba un préstamo**.

In [ ]:
# ▶️ Generamos un historial de clientes ficticio
rng = np.random.default_rng(0)
n = 300
clientes = pd.DataFrame({
    "ingreso_mensual": rng.integers(8_000, 60_000, n),
    "deuda_pct":       rng.integers(0, 80, n),        # % del ingreso que ya va a deudas
    "tiene_aval":      rng.integers(0, 2, n),          # 1 = sí, 0 = no
})
# La "regla real" que el banco siguió en el pasado (el modelo NO la conoce)
clientes["aprobado"] = (
    ((clientes.ingreso_mensual > 20_000) & (clientes.deuda_pct <= 40)) |
    ((clientes.ingreso_mensual <= 20_000) & (clientes.tiene_aval == 1))
).astype(int)

clientes.head(8)

Ahora le pedimos a un árbol que **descubra las preguntas por sí solo**, mirando únicamente los ejemplos.

In [ ]:
X = clientes[["ingreso_mensual", "deuda_pct", "tiene_aval"]]
y = clientes["aprobado"]

arbol_banco = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print(export_text(arbol_banco, feature_names=list(X.columns), class_names=["Rechazar", "Aprobar"]))

Fíjate en lo poderoso de esto: **puedes leer el modelo**. Si un cliente pregunta "¿por qué me rechazaron?", basta con seguir su camino por el árbol. Muy pocos modelos permiten eso.

> 📝 Los cortes que encontró el árbol (p. ej. `ingreso_mensual <= 20xxx`) no son exactamente los de la regla real, pero quedan muy cerca. **Lo aprendió solo a partir de los ejemplos.**
>
> 📝 Fíjate también en que la primera pregunta del árbol es la **deuda**, no el ingreso. El orden de las preguntas puede ser distinto al de la regla original y aun así llegar a las mismas decisiones.

### ✋ Ejercicio 2 — Sé el árbol
Usando el árbol que se imprimió arriba, decide **a mano** qué le pasa a cada cliente:

| Cliente | Ingreso | Deuda % | Aval |
|---|---|---|---|
| Ana | 35,000 | 25 | No |
| Beto | 12,000 | 10 | No |
| Carla | 15,000 | 60 | Sí |

🔮 Escribe tus predicciones aquí y luego ejecuta la celda siguiente para comprobarlas.

✍️ *Tus predicciones:* Ana → ___ · Beto → ___ · Carla → ___

In [ ]:
nuevos = pd.DataFrame({
    "ingreso_mensual": [35_000, 12_000, 15_000],
    "deuda_pct":       [25, 10, 60],
    "tiene_aval":      [0, 0, 1],
}, index=["Ana", "Beto", "Carla"])

nuevos["decision"] = np.where(arbol_banco.predict(nuevos) == 1, "✅ Aprobar", "❌ Rechazar")
nuevos

<details><summary><b>👀 Ver respuesta</b></summary>

- **Ana:** deuda 25 % (≤ 40.5) → ingreso alto → ✅ Aprobar.
- **Beto:** deuda 10 % (≤ 40.5) → ingreso bajo → sin aval → ❌ Rechazar. Aunque casi no tiene deudas, sin ingreso suficiente ni aval no basta.
- **Carla:** deuda 60 % (> 40.5) → ingreso bajo → con aval → ✅ Aprobar, aunque su deuda es alta.

Lección importante: **cada rama hace sus propias preguntas.** A Ana nunca le preguntan por el aval; a Carla sí.
</details>

### 📖 Clasificación y regresión
- **Árbol de clasificación:** la hoja dice una **categoría** ("Aprobar"). Es la clase mayoritaria de los ejemplos que cayeron en esa hoja.
- **Árbol de regresión:** la hoja dice un **número** ("precio estimado: $2.3 M"). Es el promedio de los ejemplos que cayeron en esa hoja.

---
## 3. 📖 Cómo aprende: buscar la pregunta que más "ordena"

El árbol no conoce las preguntas de antemano: **las descubre a partir de los datos**. Para eso necesita medir qué tan "ordenado" queda un grupo.

### La idea de pureza
Imagina un cajón con calcetines. Está **puro** si solo tiene un color, e **impuro** si están revueltos.

```
Cajón impuro:  🔴🔵🔴🔵🔵🔴🔵🔴    (50 % / 50 % → máximo desorden)
Cajón puro:    🔴🔴🔴🔴🔴🔴🔴🔴    (100 % rojo → cero desorden)
```

Para medir el desorden se usa un **criterio de impureza**. El más común se llama **Gini**:
- vale **0** cuando el grupo es puro;
- es **máximo** (0.5 con dos clases) cuando está mezclado a partes iguales.

No necesitas la fórmula: la celda de abajo la calcula por ti.

In [ ]:
def impureza(grupo):
    # Gini: probabilidad de equivocarte si etiquetas un elemento al azar según las proporciones del grupo
    total = len(grupo)
    return 1 - sum((c / total) ** 2 for c in Counter(grupo).values())

print("Cajón puro  :", impureza("🔴🔴🔴🔴🔴🔴🔴🔴"))
print("Cajón 50/50 :", impureza("🔴🔵🔴🔵🔵🔴🔵🔴"))

### 🔮 Ejercicio 3 — Ordena los cajones
Sin ejecutar nada todavía, ordena estos cajones del **más puro** al **más impuro**:

- A: `🔴🔴🔴🔴🔴🔴🔴🔵`
- B: `🔴🔴🔴🔴🔵🔵🔵🔵`
- C: `🔴🔴🔴🔴🔴🔴🔵🔵`

✍️ *Tu orden:*

Luego ejecuta la celda y comprueba. ✏️ Inventa también tus propios cajones (puedes añadir 🟢 como tercer color).

In [ ]:
cajones = {
    "A": "🔴🔴🔴🔴🔴🔴🔴🔵",
    "B": "🔴🔴🔴🔴🔵🔵🔵🔵",
    "C": "🔴🔴🔴🔴🔴🔴🔵🔵",
    # ✏️ añade aquí tus cajones, por ejemplo:  "D": "🔴🔵🟢🔴🔵🟢",
}
for nombre, cajon in sorted(cajones.items(), key=lambda kv: impureza(kv[1])):
    print(f"{nombre}: {cajon}  → impureza = {impureza(cajon):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**A (0.219) → C (0.375) → B (0.5)**. Cuanto más dominante es un color, más puro es el cajón. Con tres colores repartidos por igual la impureza sube aún más (0.667): hay más formas de equivocarse.
</details>

### 📖 El algoritmo, paso a paso
1. Toma **todos** los datos de entrenamiento en el nodo raíz.
2. Prueba **todas** las preguntas posibles: cada característica y cada punto de corte.
3. Elige la pregunta que deja los grupos resultantes **más puros** (en promedio, ponderando por tamaño).
4. Divide los datos en dos grupos según la respuesta.
5. **Repite** los pasos 2 a 4 en cada grupo, por separado.
6. Para cuando se cumpla una **condición de parada** (aquí entran los hiperparámetros).

Este proceso es **voraz** (*greedy*): en cada paso elige la mejor pregunta *en ese momento*, sin mirar hacia adelante.

> 💡 **Diferencia con la Lección 3.3:** un árbol **no usa descenso de gradiente**. No ajusta perillas poco a poco; construye su estructura dividiendo los datos. Por eso no tiene tasa de aprendizaje ni épocas.

Vamos a hacer de árbol nosotros mismos.

### ✋ Ejercicio 4 — Construye un árbol a mano 🍦
Una heladería quiere predecir si un día **venderá mucho** helado.

In [ ]:
heladeria = pd.DataFrame({
    "calor":         ["Sí", "Sí", "No", "No", "Sí", "No", "Sí", "No"],
    "fin_de_semana": ["Sí", "No", "Sí", "No", "Sí", "No", "No", "No"],
    "vendio_mucho":  ["✅", "✅", "✅", "❌", "✅", "❌", "✅", "❌"],
}, index=[f"Día {i}" for i in range(1, 9)])
heladeria

Si solo pudieras hacer **una** pregunta, ¿cuál elegirías: "¿Hace calor?" o "¿Es fin de semana?"?

🔮 ✍️ *Mi apuesta:*

La función `probar_pregunta` divide los días según la respuesta y mide la impureza de cada grupo. Ejecútala para las dos preguntas.

In [ ]:
def probar_pregunta(datos, columna, objetivo="vendio_mucho"):
    print(f"Pregunta: ¿{columna}?")
    total, ponderada = len(datos), 0
    for respuesta, grupo in datos.groupby(columna):
        etiquetas = "".join(grupo[objetivo])
        imp = impureza(etiquetas)
        ponderada += len(grupo) / total * imp
        print(f"   {respuesta:>2} → {etiquetas:<10} impureza = {imp:.3f}")
    print(f"   Impureza promedio después de dividir: {ponderada:.3f}\n")

probar_pregunta(heladeria, "calor")
probar_pregunta(heladeria, "fin_de_semana")

<details><summary><b>👀 Ver respuesta</b></summary>

**"¿Hace calor?"** gana: impureza promedio **0.188** contra **0.300**. Deja un grupo totalmente puro (4 ✅ de 4) y el otro bastante ordenado (3 ❌ y 1 ✅). "¿Es fin de semana?" también deja un grupo puro, pero el otro queda muy revuelto.
</details>

Ahora dibuja (en papel o aquí abajo) el árbol completo con **dos** preguntas. ¿Clasifica bien los 8 días?

✍️ *Tu árbol:*
```
¿...?
 ├── ...
 └── ...
```
Compáralo con el que encuentra la computadora:

In [ ]:
X_h = (heladeria[["calor", "fin_de_semana"]] == "Sí").astype(int)
y_h = heladeria["vendio_mucho"]
arbol_helado = DecisionTreeClassifier(random_state=0).fit(X_h, y_h)
print(export_text(arbol_helado, feature_names=["calor", "fin_de_semana"]))

<details><summary><b>👀 Ver respuesta</b></summary>

```
¿Hace calor?
 ├── Sí → ✅
 └── No → ¿Es fin de semana?
           ├── Sí → ✅
           └── No → ❌
```
Clasifica los 8 días correctamente. En la salida de la computadora, `calor <= 0.50` significa "calor = No".
</details>

---
## 4. 📖 ¿Qué pasa si nunca lo detienes?

Si dejas que el árbol crezca sin límite, seguirá haciendo preguntas hasta que **cada hoja sea perfectamente pura**, aunque eso signifique crear una hoja para un solo ejemplo raro:

```
¿...? → ¿...? → ¿...? → ¿Se llama Juan y nació un martes? → ✅ Aprobar  (1 solo ejemplo)
```

El resultado: casi 100 % de acierto en entrenamiento y un desempeño mucho peor con datos nuevos. El árbol **memorizó** en lugar de **aprender**. Es el ejemplo de manual del **sobreajuste** (Lección 4.4).

Vamos a comprobarlo con un conjunto de datos con forma de dos lunas y algo de ruido.

In [ ]:
X_lunas, y_lunas = make_moons(n_samples=400, noise=0.35, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X_lunas, y_lunas, test_size=0.3, random_state=1)

plt.scatter(*X_tr.T, c=y_tr, cmap="coolwarm", edgecolor="k", s=25)
plt.title("Datos de entrenamiento: dos 'lunas' con ruido"); plt.show()

### 🔮 Ejercicio 5 — Predice antes de ejecutar
Vamos a entrenar un árbol **sin ningún límite** (`max_depth=None`).
1. ¿Qué exactitud esperas en **entrenamiento**?
2. ¿Y en **prueba**: más alta, igual o más baja?

✍️ *Mis predicciones:*

In [ ]:
sin_limite = DecisionTreeClassifier(max_depth=None, random_state=0).fit(X_tr, y_tr)
print(f"Exactitud en entrenamiento: {sin_limite.score(X_tr, y_tr):.0%}")
print(f"Exactitud en prueba       : {sin_limite.score(X_te, y_te):.0%}")
print(f"Número de hojas           : {sin_limite.get_n_leaves()}  (¡para {len(X_tr)} ejemplos!)")

<details><summary><b>👀 Ver respuesta</b></summary>

Entrenamiento: **100 %**. Prueba: claramente más baja. El árbol creó decenas de hojas para acomodar cada punto ruidoso: **memorizó**. Esa brecha entre entrenamiento y prueba es la firma del sobreajuste.
</details>

👉 **Por eso, controlar el crecimiento del árbol es el trabajo central de sus hiperparámetros.**

---
## 5. 📖 Hiperparámetros: ¿cuándo dejo de preguntar?

Casi todos los hiperparámetros de un árbol responden a la misma pregunta: **¿cuándo dejo de dividir?**

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `max_depth` | Máximo de preguntas en un camino | Más complejo → riesgo de **sobreajuste** | Más simple → riesgo de **subajuste** | 3–10 |
| `min_samples_split` | Mínimo de ejemplos para poder dividir un nodo | Más simple | Más complejo | 2–20 |
| `min_samples_leaf` | Mínimo de ejemplos en cada hoja | Más simple, hojas más "confiables" | Permite hojas de 1 ejemplo (memorización) | 1–10 (o ~1 % de los datos) |
| `max_leaf_nodes` | Número máximo de hojas | Más complejo | Más simple | Sin límite / 10–100 |
| `max_features` | Cuántas características se consideran en cada división | Usa más información | Más aleatorio (clave en Random Forest, M6) | Todas |
| `criterion` | Cómo se mide la impureza | — | — | `gini` / `squared_error` |
| `ccp_alpha` | Poda: castigo por cada hoja extra | Más podado y simple | Más grande | 0.0 (ajustar con validación cruzada) |
| `class_weight` | Importancia de cada clase | — | — | `None`, o `balanced` si hay desbalance |

### 🧠 Regla mnemotécnica
> **Los `max_` hacen crecer el árbol; los `min_` lo frenan.**

Veamos qué significa esto **visualmente**. Un árbol divide el espacio en **rectángulos**: cada pregunta mira una sola característica a la vez, así que sus fronteras son siempre líneas verticales u horizontales.

In [ ]:
def dibujar_frontera(modelo, X, y, ax, titulo):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 300),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 300))
    zz = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap="coolwarm")
    ax.scatter(*X.T, c=y, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(titulo, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

def comparar(parametro, valores):
    fig, axes = plt.subplots(1, len(valores), figsize=(4 * len(valores), 3.8))
    for ax, v in zip(axes, valores):
        m = DecisionTreeClassifier(**{parametro: v}, random_state=0).fit(X_tr, y_tr)
        dibujar_frontera(m, X_tr, y_tr, ax,
                         f"{parametro}={v}\ntrain {m.score(X_tr, y_tr):.0%} · test {m.score(X_te, y_te):.0%}")
    plt.tight_layout(); plt.show()

comparar("max_depth", [1, 4, None])

- **`max_depth=1`** → una sola pregunta: **subajuste**. Ignora la forma de las lunas.
- **`max_depth=4`** → captura el patrón sin memorizar el ruido.
- **`max_depth=None`** → islas diminutas alrededor de puntos sueltos: **sobreajuste**.

### 🔮 Ejercicio 6 — La regla mnemotécnica al revés
Ahora variaremos `min_samples_leaf` con los valores **1, 10 y 80**.
- ¿Cuál de los tres gráficos crees que se parecerá al de `max_depth=None`?
- ¿Cuál se parecerá al de `max_depth=1`?

✍️ *Mis predicciones:*

In [ ]:
comparar("min_samples_leaf", [1, 10, 80])

<details><summary><b>👀 Ver respuesta</b></summary>

- `min_samples_leaf=1` ≈ `max_depth=None` → sobreajuste (hojas de un solo punto).
- `min_samples_leaf=80` ≈ `max_depth=1` → subajuste (pocas hojas muy grandes).
- `min_samples_leaf=10` → el punto intermedio.

Es la regla mnemotécnica en acción: **subir un `min_` simplifica, subir un `max_` complica.**
</details>

### ✏️ Ejercicio 7 — Encuentra el punto justo
Ahora con datos reales: diagnóstico de cáncer de mama (569 pacientes, 30 características). La celda dibuja la exactitud en entrenamiento y en prueba para cada profundidad.

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, random_state=42)

PARAMETRO = "max_depth"     # ✏️ prueba también "min_samples_leaf"
VALORES = range(1, 16)       # ✏️ para min_samples_leaf usa, por ejemplo, range(1, 60, 4)

train_acc, test_acc = [], []
for v in VALORES:
    m = DecisionTreeClassifier(**{PARAMETRO: v}, random_state=42).fit(Xc_tr, yc_tr)
    train_acc.append(m.score(Xc_tr, yc_tr))
    test_acc.append(m.score(Xc_te, yc_te))

plt.plot(list(VALORES), train_acc, "o-", label="Entrenamiento")
plt.plot(list(VALORES), test_acc, "s-", label="Prueba")
plt.xlabel(PARAMETRO); plt.ylabel("Exactitud"); plt.legend(); plt.grid(alpha=.3)
plt.title("¿Dónde empieza el sobreajuste?"); plt.show()

Responde:
1. ¿A partir de qué profundidad la curva de entrenamiento llega casi a 1.0? ¿Qué hace la curva de prueba en ese punto?
2. ¿Qué `max_depth` elegirías?
3. ✏️ Cambia a `PARAMETRO = "min_samples_leaf"` y vuelve a ejecutar. ¿Por qué la gráfica se ve "al revés"?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Desde profundidad 4 el entrenamiento ya ronda el 99–100 %, mientras que la prueba alcanza su mejor valor hacia profundidad 3 y luego se estanca o baja un poco: el árbol empieza a memorizar.
2. Una profundidad baja-media (≈3–5), donde la prueba es buena y la brecha con entrenamiento es pequeña. **Ojo:** elegir mirando el conjunto de prueba es hacer trampa. En la práctica se elige con **validación cruzada** (Lección 4.6) y el conjunto de prueba se usa una sola vez al final.
3. Porque `min_samples_leaf` funciona al revés: valores **bajos** = árbol complejo (sobreajuste, a la izquierda) y valores **altos** = árbol simple (subajuste, a la derecha).
</details>

### 📖 Pre-poda vs. post-poda
- **Pre-poda:** detener el crecimiento *mientras* se construye (`max_depth`, `min_samples_leaf`…). Es como decir "no hagas más de 5 preguntas".
- **Post-poda:** dejar crecer el árbol completo y luego **recortar** las ramas que no aportan (`ccp_alpha`). Es como un jardinero que deja crecer el arbusto y luego quita lo que sobra.

▶️ Primero, un árbol pequeño con **pre-poda** (`max_depth=3`) sobre el diagnóstico de cáncer:

In [ ]:
arbol_medico = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xc_tr, yc_tr)
plt.figure(figsize=(18, 7))
plot_tree(arbol_medico, feature_names=Xc.columns, class_names=["maligno", "benigno"],
          filled=True, fontsize=9, impurity=False)
plt.show()
print(f"Exactitud en prueba con solo 3 niveles de preguntas: {arbol_medico.score(Xc_te, yc_te):.0%}")

### ✋ Ejercicio 8
¿Cuál es la **primera pregunta** que hace el árbol? ¿Podrías explicarle este árbol a un médico en dos frases?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

La primera pregunta es **`mean concave points <= 0.05`**: qué tan pronunciadas son, en promedio, las "hendiduras" del contorno de las células. Para un médico: *"Si el contorno de las células es casi liso, el tumor suele ser benigno, salvo que sea muy grande. Si tiene muchas hendiduras, el árbol revisa la textura y el tamaño, y la mayoría de esos casos sale maligno."*

Un árbol de 3 niveles (8 hojas) ya acierta ~96 % y **se puede explicar**. Esa es su gran ventaja.
</details>

### 🔮 Ejercicio 9 — Poda con tijeras (`ccp_alpha`)
Ahora la **post-poda**: dejamos crecer el árbol completo y luego `ccp_alpha` recorta las ramas que aportan poco. Cuanto más alto el valor, más fuerte la tijera.

Antes de ejecutar: si subimos `ccp_alpha` de 0 a 0.05, ¿qué pasa con el número de hojas? ¿Y con la exactitud de prueba: sube siempre, baja siempre o ninguna de las dos?

✍️ *Mi predicción:*

In [ ]:
for alpha in [0.0, 0.005, 0.01, 0.02, 0.05]:   # ✏️ prueba otros valores
    m = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42).fit(Xc_tr, yc_tr)
    print(f"ccp_alpha={alpha:<6} hojas={m.get_n_leaves():>3}  profundidad={m.get_depth()}  "
          f"train={m.score(Xc_tr, yc_tr):.1%}  test={m.score(Xc_te, yc_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Las hojas bajan siempre (16 → 10 → 8 → 4 → 2). La exactitud de prueba **no** se mueve en una sola dirección: primero mejora un poco (≈95 % → ≈96 %) porque la poda quita ramas que solo memorizaban ruido, y luego cae (≈90 %) cuando la poda es tan fuerte que el árbol queda en subajuste. Es la misma curva en "U" de siempre, vista desde otra perilla. El mejor `ccp_alpha` se elige con validación cruzada.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Entrenamiento ≈ 100 % y prueba mucho menor | Árbol sin límites → sobreajuste | Bajar `max_depth`, subir `min_samples_leaf`, usar `ccp_alpha` |
| Exactitud baja en entrenamiento **y** en prueba | Árbol demasiado restringido → subajuste | Subir `max_depth`, bajar `min_samples_leaf` |
| El árbol cambia por completo al reentrenarlo con datos casi iguales | Inestabilidad natural de los árboles | Usar un ensamble (Random Forest, M6) |
| Siempre predice la clase mayoritaria | Datos desbalanceados | `class_weight='balanced'`, cambiar la métrica (Lección 4.2) |
| Una variable como "ID de cliente" sale como la más importante | El árbol memoriza identificadores (posible **fuga de datos**) | Eliminar columnas identificadoras (Lección 2.6) |
| Predicciones absurdas fuera del rango visto (regresión) | Los árboles **no extrapolan** | Usar otro modelo si necesitas predecir fuera de rango |

El último punto sorprende a mucha gente. Comprobémoslo.

### 🔮 Ejercicio 10 — La mansión
Entrenamos un árbol de **regresión** para predecir el precio de casas de entre 50 y 300 m². Luego le preguntamos por una mansión de **1,000 m²**.

¿Qué precio crees que predecirá? ✍️ *Mi apuesta:*

In [ ]:
m2 = rng.uniform(50, 300, 200)
precio = m2 * 15_000 + rng.normal(0, 200_000, 200)       # ~ $15,000 por m²

arbol_casas = DecisionTreeRegressor(max_depth=5).fit(m2.reshape(-1, 1), precio)

for tam in [100, 250, 300, 1_000]:
    pred = arbol_casas.predict([[tam]])[0]
    print(f"{tam:>5} m² → predicción: ${pred:>12,.0f}   (precio 'real' ≈ ${tam * 15_000:,.0f})")

<details><summary><b>👀 Ver respuesta</b></summary>

La mansión recibe **aproximadamente el mismo precio que una casa de 300 m²**. Cada hoja predice el promedio de los ejemplos que vio, y la mansión cae en la hoja de "las casas más grandes". El árbol **no puede imaginar** valores que nunca vio. Una regresión lineal (Lección 5.1), en cambio, sí extrapolaría la tendencia.
</details>

---
## 7. 📋 Ficha del modelo

| | Árbol de decisión |
|---|---|
| **Tipo** | Supervisado: clasificación y regresión |
| **Intuición** | Una secuencia de preguntas de sí/no que llevan a una respuesta |
| **Cómo aprende** | Elige de forma voraz la división que más reduce la impureza y repite el proceso |
| **Hiperparámetros clave** | `max_depth`, `min_samples_leaf`, `ccp_alpha` |
| **✅ Fortalezas** | Interpretable · No necesita escalar los datos · Captura relaciones no lineales · Rápido |
| **❌ Debilidades** | Sobreajusta con facilidad · Inestable · Fronteras "en escalera" · No extrapola |
| **Úsalo cuando…** | Necesitas explicar las decisiones, quieres una línea base rápida o vas a construir un ensamble |
| **Evítalo cuando…** | Buscas la máxima precisión con un único modelo (usa Random Forest o boosting, M6) |

## 8. 🧠 Resumen en 5 puntos
1. Un árbol de decisión predice haciendo **preguntas de sí/no** sobre las características.
2. Aprende eligiendo, en cada paso, la pregunta que deja los grupos **más puros**.
3. Sin límites, un árbol **memoriza** los datos: el sobreajuste es su enemigo natural.
4. Sus hiperparámetros deciden **cuándo dejar de preguntar**: los `max_` lo hacen crecer y los `min_` lo frenan.
5. Es muy **interpretable**, pero **inestable**; por eso es la pieza base de los ensambles del Módulo 6.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Tu árbol tiene 99 % de exactitud en entrenamiento y 71 % en prueba. ¿Qué cambio tiene **más** sentido?
- a) Subir `max_depth` de 8 a 20
- b) Bajar `min_samples_leaf` de 5 a 1
- c) Subir `min_samples_leaf` de 1 a 20
- d) Cambiar `criterion` de `gini` a `entropy`

**2.** ¿Cuál de estas afirmaciones es **falsa**?
- a) Los árboles no necesitan escalar los datos
- b) Los árboles se entrenan con descenso de gradiente
- c) Un árbol puede usarse para regresión
- d) Un árbol es fácil de interpretar

**3.** Tienes 10,000 ejemplos y pones `min_samples_leaf = 5000`. ¿Cuántas hojas puede tener el árbol como máximo?
- a) 1 · b) 2 · c) 5,000 · d) Sin límite

**4.** Un árbol con `max_depth = 1` (una sola pregunta) se llama *tocón* (*stump*). ¿Qué esperas de él por sí solo?
- a) Sobreajuste · b) Subajuste · c) El mejor desempeño posible · d) Un error de scikit-learn

**5.** Un árbol sin límites y uno con `ccp_alpha = 0.02` se entrenan con los mismos datos. ¿Cuál tendrá **más** hojas?
- a) El de `ccp_alpha = 0.02` · b) El que no tiene límites · c) Los dos igual

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** La brecha enorme es sobreajuste. Subir `min_samples_leaf` simplifica el árbol; a) y b) lo complican; d) casi no cambia nada.
2. **b)** Los árboles se construyen con un algoritmo voraz de divisiones, no con gradientes.
3. **b)** Cada hoja necesita al menos 5,000 ejemplos, así que caben como máximo dos. Esperarías **subajuste**.
4. **b)** Una sola pregunta casi nunca basta. Pero es la pieza base de **AdaBoost** (M6), que combina cientos de tocones.
5. **b)** `ccp_alpha` castiga cada hoja extra: cuanto más alto, más poda.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** ¿Por qué un árbol **no** necesita que escales las características (ingreso en pesos, edad en años), mientras que KNN (Lección 5.3) **sí** lo necesita?

**B. Caso de análisis 💼** Un banco quiere un modelo para aprobar créditos. El regulador exige que **toda** denegación pueda explicarse al cliente. El equipo propone:
- **Opción A:** un árbol con `max_depth = 4` y 84 % de exactitud.
- **Opción B:** un ensamble de 500 árboles con 89 % de exactitud.

¿Cuál recomendarías y por qué? ¿Qué información adicional pedirías antes de decidir? (Piensa en el costo de cada tipo de error, Lección 4.8.)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Un árbol compara **una característica a la vez** contra un umbral ("¿ingreso > 20,000?"). Cambiar la escala solo mueve el umbral, no el orden de los datos. KNN calcula **distancias** que mezclan todas las características, así que la variable con números más grandes dominaría a las demás.

**B.** No hay una única respuesta correcta; importa el razonamiento. Un buen argumento: la regulación exige explicar cada denegación, así que la **Opción A** cumple directamente. Para la B existen técnicas de explicabilidad (SHAP, Lección 7.6), pero son aproximaciones y podrían no satisfacer al regulador. Información útil:
- ¿Qué cuesta más, aprobar a un mal pagador o rechazar a un buen cliente?
- ¿Cómo se comparan en **precisión** y **recall**, no solo en exactitud?
- ¿Acepta el regulador explicaciones post-hoc?
</details>

### 9.3 🧩 Práctica integradora: "¿Qué clientes se van a ir?"

Una empresa de telefonía te entrega el historial de 2,000 clientes y una columna `se_fue` (1 = canceló su contrato). Quieren un árbol que les diga **qué clientes están en riesgo y por qué**.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng_tel = np.random.default_rng(7)
n_tel = 2_000
tel = pd.DataFrame({
    "meses_cliente":    rng_tel.integers(1, 72, n_tel),
    "llamadas_soporte": rng_tel.poisson(1.5, n_tel),
    "cargo_mensual":    rng_tel.integers(200, 1_500, n_tel),
    "contrato_anual":   rng_tel.integers(0, 2, n_tel),     # 1 = sí
})
riesgo = (-2.0 + 1.1 * tel.llamadas_soporte - 0.07 * tel.meses_cliente
          + 0.0025 * tel.cargo_mensual - 2.2 * tel.contrato_anual)
tel["se_fue"] = (rng_tel.random(n_tel) < 1 / (1 + np.exp(-riesgo))).astype(int)
# El sistema de la empresa exportó la tabla así (no cambies estas dos líneas):
tel = tel.sort_values("se_fue", kind="stable").reset_index(drop=True)
tel.insert(0, "id_cliente", np.arange(100_001, 100_001 + n_tel))
tel = tel.sample(frac=1, random_state=0).reset_index(drop=True)

X_tel = tel.drop(columns="se_fue")
y_tel = tel["se_fue"]
X_tel_tr, X_tel_te, y_tel_tr, y_tel_te = train_test_split(
    X_tel, y_tel, test_size=0.25, stratify=y_tel, random_state=0)

print(f"Clientes que se fueron: {y_tel.mean():.0%}")
tel.head()

**Paso 2 🔮 — Primer intento.** Entrenamos un árbol sin límites con **todas** las columnas. Ya sabes que en entrenamiento dará ~100 %. ¿Y en **prueba**?

✍️ *Mi apuesta:*

In [ ]:
arbol_tel = DecisionTreeClassifier(random_state=0).fit(X_tel_tr, y_tel_tr)
print(f"Entrenamiento: {arbol_tel.score(X_tel_tr, y_tel_tr):.1%}")
print(f"Prueba       : {arbol_tel.score(X_tel_te, y_tel_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡**~100 % también en prueba**! Por lo que sabes de la Sección 4, esperabas una caída. Un resultado *demasiado bueno* con un problema difícil (predecir conducta humana) es una **señal de alerta**, no un motivo de celebración. Algo huele mal… 🕵️
</details>

**Paso 3 ✋ — Encuentra al culpable.** `feature_importances_` dice cuánto usó el árbol cada columna (suman 1). Ejecuta y explica qué está pasando.

In [ ]:
pd.Series(arbol_tel.feature_importances_, index=X_tel.columns).sort_values(ascending=False)

✍️ *¿Qué está pasando?*

<details><summary><b>👀 Ver respuesta</b></summary>

El árbol usa **solo `id_cliente`**. El sistema numeró los clientes *después* de ordenarlos por `se_fue`, así que todos los que se fueron tienen los ID más altos: una sola pregunta (`id_cliente > 101,4xx`) separa perfectamente. Es **fuga de datos** (Lección 2.6): la columna contiene la respuesta "por accidente". Con clientes nuevos, el ID no dirá nada y el modelo será inútil.

Regla: **los identificadores nunca son características.** Es la fila de "ID de cliente" de la tabla de errores comunes.
</details>

**Paso 4 ✏️ — Quita la columna tramposa.** Completa la línea marcada y vuelve a entrenar el árbol sin límites. ¿Qué diagnóstico le das ahora?

In [ ]:
COLUMNAS_A_QUITAR = ["id_cliente"]   # ✏️ (ya está puesta; compruébalo tú)
X_tel_tr2 = X_tel_tr.drop(columns=COLUMNAS_A_QUITAR)
X_tel_te2 = X_tel_te.drop(columns=COLUMNAS_A_QUITAR)

arbol_tel = DecisionTreeClassifier(random_state=0).fit(X_tel_tr2, y_tel_tr)
print(f"Entrenamiento: {arbol_tel.score(X_tel_tr2, y_tel_tr):.1%}")
print(f"Prueba       : {arbol_tel.score(X_tel_te2, y_tel_te):.1%}")
print(f"Hojas        : {arbol_tel.get_n_leaves()}")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

Entrenamiento ~100 %, prueba ~75 %, cientos de hojas: **sobreajuste** de manual. Ahora el resultado es honesto, pero el árbol memoriza. Toca ponerle freno.
</details>

**Paso 5 ✏️ — Elige `max_depth` sin hacer trampa.** Esta vez **no miramos el conjunto de prueba**: usamos validación cruzada (Lección 4.6) solo con los datos de entrenamiento. Cambia `VALORES` si quieres explorar otro rango.

In [ ]:
VALORES = range(1, 11)   # ✏️

for v in VALORES:
    cv = cross_val_score(DecisionTreeClassifier(max_depth=v, random_state=0),
                         X_tel_tr2, y_tel_tr, cv=5).mean()
    print(f"max_depth={v:>2}  →  exactitud media en validación cruzada: {cv:.3f}  " + "█" * int((cv - 0.6) * 100))

✍️ *Mi `max_depth` elegido:*

<details><summary><b>👀 Ver respuesta</b></summary>

El máximo está alrededor de **`max_depth` = 4–5 (≈0.80)**. Con más profundidad la validación cruzada empieza a bajar (sobreajuste); con menos, el árbol se queda corto (subajuste). Si dos valores empatan casi, elige el **más simple**: será más fácil de explicar.
</details>

**Paso 6 ▶️ — Examen final, una sola vez.** Ahora sí usamos el conjunto de prueba. Lo comparamos con un modelo "tonto" que **siempre dice que el cliente se queda**.

In [ ]:
MAX_DEPTH = 5   # ✏️ pon aquí el valor que elegiste

final = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=0).fit(X_tel_tr2, y_tel_tr)
print(f"Árbol (max_depth={MAX_DEPTH}) en prueba : {final.score(X_tel_te2, y_tel_te):.1%}")
print(f"Modelo 'todos se quedan'      : {1 - y_tel_te.mean():.1%}")

# Un árbol de 3 niveles para presentárselo al gerente
print()
print(export_text(DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tel_tr2, y_tel_tr),
                  feature_names=list(X_tel_tr2.columns), class_names=["se queda", "se va"]))

✋ Responde:
1. ¿Cuántos puntos le gana tu árbol al modelo "todos se quedan"? ¿Te parece mucho o poco?
2. Usando el árbol de 3 niveles, escribe **dos reglas de negocio** que le darías al gerente ("Los clientes que… tienden a irse").
3. ¿Qué harías con esas reglas al día siguiente?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Unos 4 puntos (≈78 % vs. ≈73 %). Parece poco, pero el modelo "tonto" **nunca** detecta a nadie que se va, y ese es justo el objetivo. Para medirlo bien necesitarás *recall* y precisión (Lección 4.2), no solo exactitud.
2. Por ejemplo:
   - "Los clientes **nuevos** (≤ 25 meses), **sin contrato anual** y con **2 o más llamadas a soporte** tienden a irse."
   - "Los clientes antiguos también se van si llaman **3 o más veces** a soporte y pagan **más de ~$1,100** al mes."
3. Algo accionable: ofrecer contrato anual a clientes nuevos, atender de forma prioritaria a quien ya llamó dos veces a soporte, revisar planes caros. **Esto es lo que un árbol hace mejor: convertir datos en reglas que una persona puede usar.**
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *A Visual Introduction to Machine Learning* (R2D3): árboles de decisión explicados visualmente.
- 📺 **Video:** StatQuest, "Decision and Classification Trees, Clearly Explained" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Decision Trees*.

---
⬅️ **Anterior:** [5.3 K vecinos más cercanos (KNN): "dime con quién andas"](5.3-k-vecinos-mas-cercanos.ipynb) · ➡️ **Siguiente:** [5.5 Máquinas de vectores de soporte (SVM): la calle más ancha](5.5-maquinas-de-vectores-de-soporte.ipynb)